# Badge 1: DORA checks

Every DORA step from the course, in order. Each check has two cells:

1. **Local check**: the course's check query with the external `GRADER` call removed. It runs entirely in this account and sends nothing.
2. **DORA (exact course code)**: verbatim and unedited, as the course requires (DORA audits a hash of the code).

⚠️ **Running a DORA cell, or the Setup cells, sends data outside this account.** The GRADER is an external function that calls Snowflake Education's grading service (`https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora`) with the step result plus `current_account`, `current_account_name`, `current_statement` and a timestamp. Run only the local checks unless you intend to submit.

DORA code uses the real `ACCOUNTADMIN`, as the course specifies. Each local check names the lesson notebook it follows.

## Setup (course Lesson 6, 🎭 Meet DORA!)
The course: *You don't have to understand what it means to “create an API Integration,” you just need to run the code below (using the ACCOUNTADMIN ROLE!).*

In [ ]:
%%sql -r dora_api_integration
use role accountadmin;

create or replace api integration dora_api_integration
api_provider = aws_api_gateway
api_aws_role_arn = 'arn:aws:iam::321463406630:role/snowflakeLearnerAssumedRole'
enabled = true
api_allowed_prefixes = ('https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora');

In [ ]:
%%sql -r dora_grader_function
use role accountadmin;  

create or replace external function util_db.public.grader(
      step varchar
    , passed boolean
    , actual integer
    , expected integer
    , description varchar)
returns variant
api_integration = dora_api_integration 
context_headers = (current_timestamp, current_account, current_statement, current_account_name) 
as 'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora/grader'
; 

### Is the GRADER Function working?

In [ ]:
%%sql -r dora_is_working
use role accountadmin;
use database util_db; 
use schema public; 

select grader(step, (actual = expected), actual, expected, description) as graded_results from
(SELECT 
 'DORA_IS_WORKING' as step
 ,(select 123) as actual
 ,123 as expected
 ,'Dora is working!' as description
); 